# Kohl et al. (1995) — UVCS Implementation / SOHO 자외선 코로나그래프 분광기 구현

**Paper**: J. L. Kohl et al., *The Ultraviolet Coronagraph Spectrometer for the Solar and Heliospheric Observatory*, Solar Physics **162**, 313–356 (1995). DOI: 10.1007/BF00733433

This notebook reproduces four of the central UVCS measurement concepts using a forward / inverse modelling approach with synthetic data:

1. **Coronagraph occultation geometry** — external + internal occulter shadow boundaries and the 1.2–10 R⊙ vignetting profile (Eq. 3 of the paper).
2. **H I Lyα and O VI 1032/1037 line profiles** under the Doppler-dimming/pumping framework (Eq. 2 + Sect. 2).
3. **Ion temperature anisotropy** ($T_\perp$ vs $T_\parallel$) recovered from line-width fitting in two LOS geometries.
4. **Solar-wind outflow speed retrieval** from the O VI 1032/1037 doublet ratio with C II pumping.

이 노트북은 UVCS 논문의 네 가지 핵심 측정 개념(차폐 기하, 라인 프로파일 forward 모델, 이온 온도 비등방성 적합, OVI 비를 이용한 흐름 속도 역산)을 NumPy/SciPy/Matplotlib만으로 합성 데이터에 대해 구현한다. 모든 수식은 논문의 식 (2)–(3) 및 Sect. 2에서 직접 차용한다.

**Kernel**: `study-with-ai`

In [ ]:
"""Imports and global plot configuration for the UVCS implementation notebook."""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, brentq
from scipy.integrate import quad

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants in SI/CGS hybrid units commonly used in solar UV spectroscopy.
K_B = 1.380649e-23      # Boltzmann constant [J/K]
M_PROTON = 1.67262e-27  # Proton mass [kg]
M_OXYGEN = 16.0 * M_PROTON
C_LIGHT = 2.99792458e5  # Speed of light [km/s]
R_SUN_KM = 6.957e5      # Solar radius [km]

rng = np.random.default_rng(seed=20260425)
print('Environment ready: NumPy', np.__version__)

## Part 1: Coronagraph Occultation Geometry / 코로나그래프 차폐 기하

UVCS uses an **external serrated occulter** to geometrically block the solar disk and an **internal occulter** at the intermediate focal plane to block the diffracted bright rim that the external occulter still scatters into the telescope mirror (paper Sect. 4.2, Fig. 4). The unvignetted mirror area available for a line-of-sight (LOS) at heliocentric distance $r$ (in $R_\odot$) follows Eq. (3):

$$ A(r) = h\,D\,\tan\!\left[\tfrac{16}{60}(r-1.2)\right] - b $$

where $h = 72$ mm is the mirror height, $D$ is the external-occulter-to-mirror distance, $16'/(R-1.2)$ is the linear angular shadow boundary (≈$0.27°$ per $R_\odot$, the apparent solar limb angular size), and $b$ is the over-occulting (internal-occulter) bias.

UVCS는 외부 차폐(serrated occulter)로 디스크를 가리고, 내부 차폐로 외부 차폐 가장자리에서 회절된 빛을 다시 차단한다. 식 (3)은 시선 높이 $r$ ($R_\odot$ 단위)에서 거울로 들어오는 비비네팅(unvignetted) 면적이 $1.2\,R_\odot$ 부근에서는 0이고 점차 선형적으로 증가함을 나타낸다. 본 절에서는 이 기하를 시각화하여 1.25–10 $R_\odot$ 범위에서의 vignetting 프로파일을 보인다.

In [ ]:
def unvignetted_area(r_rsun, h_mm=72.0, D_mm=1100.0, b_mm2=180.0):
    """Compute the unvignetted UVCS mirror area as a function of LOS height.

    Implements Eq. (3) of Kohl et al. (1995). The angular shadow boundary is
    16 arcmin per (r - 1.2) R_sun, which equals one solar limb angular radius
    per radial step.

    Args:
        r_rsun: Heliocentric line-of-sight height in solar radii. Scalar or
            ndarray. Values below 1.2 R_sun are clipped to zero area.
        h_mm: Mirror height in millimetres (default 72 mm, paper Sect. 4.2).
        D_mm: External occulter to mirror distance in millimetres.
        b_mm2: Internal-occulter over-occulting bias area in mm^2.

    Returns:
        Unvignetted mirror area in mm^2, clipped at zero (no negative areas).
    """
    r = np.asarray(r_rsun, dtype=float)
    angle_rad = np.deg2rad((16.0 / 60.0) * (r - 1.2))
    raw_area = h_mm * D_mm * np.tan(angle_rad) - b_mm2
    return np.clip(raw_area, 0.0, None)


r_grid = np.linspace(1.25, 10.0, 400)
area_uv = unvignetted_area(r_grid, h_mm=72.0, D_mm=1100.0, b_mm2=180.0)
area_wlc = unvignetted_area(r_grid, h_mm=30.0, D_mm=1100.0, b_mm2=80.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(r_grid, area_uv, label='UV channels (h=72 mm)', lw=2)
axes[0].plot(r_grid, area_wlc, label='WLC (h=30 mm)', lw=2, linestyle='--')
axes[0].axvline(1.2, color='red', linestyle=':', label='External occulter edge')
axes[0].set_xlabel('Heliocentric height r [R_sun]')
axes[0].set_ylabel('Unvignetted mirror area [mm$^2$]')
axes[0].set_title('UVCS Unvignetted Area vs Height (Eq. 3)')
axes[0].legend()

# Cartoon ray geometry: disk, external occulter shadow, internal-occulter notch.
ax = axes[1]
theta = np.linspace(0, 2 * np.pi, 200)
ax.fill(np.cos(theta), np.sin(theta), color='gold', alpha=0.6, label='Solar disk')
for r_los in [1.5, 2.5, 5.0, 8.0]:
    ax.plot([0, 8], [r_los, r_los], lw=1.0, alpha=0.6,
            label=f'LOS @ {r_los} R_sun')
ax.axvspan(1.0, 1.2, color='gray', alpha=0.4, label='External occulter shadow')
ax.axvspan(7.5, 8.0, color='black', alpha=0.5, label='Sunlight trap')
ax.set_xlim(-1.5, 8.5)
ax.set_ylim(-1.5, 9.0)
ax.set_aspect('equal')
ax.set_xlabel('x [R_sun]')
ax.set_ylabel('y [R_sun]')
ax.set_title('Schematic UVCS geometry')
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

# Quantitative check for a few canonical heights.
for r_chk in [1.25, 1.5, 2.5, 5.0, 10.0]:
    print(f'r = {r_chk:5.2f} R_sun  ->  A = {unvignetted_area(r_chk):8.2f} mm^2')

## Part 2: H I Lyα and O VI Line Profiles under Doppler Dimming / Lyα·OVI 라인 프로파일

Each resonantly scattered coronal UV line has an emergent profile that is the convolution of the chromospheric incoming line $\phi_\text{chromo}$ with the local absorber profile $\phi_\text{abs}(\lambda)$ in the rest frame of the streaming ion (paper Sect. 2 Eq. 2). For a single LOS with bulk outflow speed $V_W$ projected onto the radial direction, the resonant intensity is reduced by the **dimming factor**

$$ D_i(V_W) = \int d\lambda\,\phi_\text{chromo}(\lambda - \lambda_0\,V_W/c)\,\phi_\text{abs}(\lambda). $$

여기서 우리는 두 라인을 모델링한다: (i) HI Lyα 1216 Å — 양성자 열운동에 지배 (FWHM ≈ 1 Å at $T_p = 10^6$ K) 그리고 (ii) OVI 1031.93 Å — 무거운 이온이라 열폭은 좁지만 비등방 운동 온도와 흐름 속도에 강하게 좌우된다. 두 색구권 입사 라인 모두 가우시안으로 가정하고, 흐름이 0–500 km/s 범위에서 변할 때 dimming 인자가 어떻게 떨어지는지를 forward 계산한다.

In [ ]:
def thermal_doppler_width_angstrom(lambda0_A, T_kelvin, m_kg):
    """Compute the 1/e thermal Doppler width Delta_lambda in angstroms.

    Args:
        lambda0_A: Rest wavelength in angstroms.
        T_kelvin: Kinetic temperature governing the LOS-projected motion.
        m_kg: Ion mass in kilograms.

    Returns:
        Doppler width Delta_lambda where the profile is exp(-((dl)/Delta)^2).
    """
    v_thermal_km_s = np.sqrt(2.0 * K_B * T_kelvin / m_kg) / 1.0e3
    return lambda0_A * v_thermal_km_s / C_LIGHT


def gaussian_profile(lam_A, lam0_A, dl_width_A):
    """Return a normalized Gaussian line profile in angstroms."""
    norm = 1.0 / (dl_width_A * np.sqrt(np.pi))
    return norm * np.exp(-((lam_A - lam0_A) / dl_width_A) ** 2)


def doppler_dimming_factor(v_wind_km_s, lam0_A, dl_chromo_A, dl_abs_A):
    """Compute D_i(V_W) for a Gaussian chromospheric incoming line and a
    Gaussian coronal absorber profile.

    For two Gaussians with widths a and b shifted by Delta, the overlap
    integral is analytic: D = 1/(sqrt(pi(a^2+b^2))) * exp(-Delta^2/(a^2+b^2)).
    """
    delta_A = lam0_A * v_wind_km_s / C_LIGHT
    sigma_sq = dl_chromo_A ** 2 + dl_abs_A ** 2
    return np.exp(-delta_A ** 2 / sigma_sq) / np.sqrt(np.pi * sigma_sq)


# H I Lyalpha — proton thermal motion at 1.5 MK.
lyA0 = 1215.67
dl_lyA_chromo = thermal_doppler_width_angstrom(lyA0, 1.5e4, M_PROTON)  # narrow chromospheric core
dl_lyA_abs = thermal_doppler_width_angstrom(lyA0, 1.5e6, M_PROTON)

# O VI 1031.93 — coronal absorber 2 MK, broadened in fast wind.
ovi0 = 1031.93
dl_ovi_chromo = thermal_doppler_width_angstrom(ovi0, 2.0e4, M_OXYGEN)
dl_ovi_abs = thermal_doppler_width_angstrom(ovi0, 2.0e6, M_OXYGEN)

print(f'Lyα 1/e widths:  chromo = {dl_lyA_chromo:.3f} Å,  absorber = {dl_lyA_abs:.3f} Å')
print(f'OVI 1/e widths:  chromo = {dl_ovi_chromo:.3f} Å,  absorber = {dl_ovi_abs:.3f} Å')

# Plot dimming factors as a function of V_W.
v_grid = np.linspace(0, 500, 250)
D_lyA = doppler_dimming_factor(v_grid, lyA0, dl_lyA_chromo, dl_lyA_abs)
D_ovi = doppler_dimming_factor(v_grid, ovi0, dl_ovi_chromo, dl_ovi_abs)

# Synthetic line profiles at V_W = 0 and 250 km/s.
lam_lyA = np.linspace(lyA0 - 2.0, lyA0 + 2.0, 600)
lam_ovi = np.linspace(ovi0 - 1.5, ovi0 + 1.5, 600)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for vw, color in zip([0, 100, 250, 400], ['k', 'tab:blue', 'tab:orange', 'tab:red']):
    profile = gaussian_profile(lam_lyA, lyA0, dl_lyA_abs)
    profile *= doppler_dimming_factor(vw, lyA0, dl_lyA_chromo, dl_lyA_abs) / D_lyA[0]
    axes[0].plot(lam_lyA, profile, color=color, label=f'V_W = {vw} km/s')
axes[0].set_title('H I Lyα coronal profile vs V_W')
axes[0].set_xlabel('Wavelength [Å]')
axes[0].set_ylabel('Normalized intensity')
axes[0].legend()

for vw, color in zip([0, 100, 250, 400], ['k', 'tab:blue', 'tab:orange', 'tab:red']):
    profile = gaussian_profile(lam_ovi, ovi0, dl_ovi_abs)
    profile *= doppler_dimming_factor(vw, ovi0, dl_ovi_chromo, dl_ovi_abs) / D_ovi[0]
    axes[1].plot(lam_ovi, profile, color=color, label=f'V_W = {vw} km/s')
axes[1].set_title('O VI 1031.93 Å profile vs V_W')
axes[1].set_xlabel('Wavelength [Å]')
axes[1].legend()

axes[2].plot(v_grid, D_lyA / D_lyA[0], label='Lyα 1216')
axes[2].plot(v_grid, D_ovi / D_ovi[0], label='OVI 1032')
axes[2].set_xlabel('Outflow speed V_W [km/s]')
axes[2].set_ylabel('Relative dimming factor D(V_W)/D(0)')
axes[2].set_title('Doppler-dimming curves')
axes[2].legend()

plt.tight_layout()
plt.show()

### 2.1 Why Lyα dims slower than O VI / 왜 Lyα가 OVI보다 천천히 dimming 되는가

The proton mass-to-oxygen-mass ratio is $1/16$, so at the same kinetic temperature the proton thermal width $v_\text{th} \propto m^{-1/2}$ is four times larger. The dimming factor falls off when the bulk shift $\lambda_0 V_W/c$ becomes comparable to the **convolved** Gaussian width $\sqrt{\Delta\lambda_\text{chromo}^2 + \Delta\lambda_\text{abs}^2}$. Because the Lyα profile is intrinsically broader, its dimming curve decays at a higher characteristic $V_W$ — typically $\sim 250$ km/s for Lyα versus $\sim 100$ km/s for O VI.

양성자/산소 질량비가 1/16이므로 같은 온도에서 양성자 열속도는 산소 이온의 4배이다. Dimming 인자가 떨어지기 시작하는 특성 흐름 속도는 두 가우시안 폭의 RSS에 비례하므로, Lyα는 ~250 km/s, OVI는 ~100 km/s 부근에서 절반으로 떨어진다. 이 차이가 UVCS가 두 라인을 동시에 측정하여 흐름·온도를 분리할 수 있는 물리적 근거다.

In [ ]:
# Find V_50, the speed at which D drops to half its V=0 value.
def half_velocity(lam0_A, dl_chromo_A, dl_abs_A):
    """Return the bulk speed at which D drops to half its zero-velocity value."""
    d0 = doppler_dimming_factor(0.0, lam0_A, dl_chromo_A, dl_abs_A)
    func = lambda v: doppler_dimming_factor(v, lam0_A, dl_chromo_A, dl_abs_A) - 0.5 * d0
    return brentq(func, 1.0, 1500.0)


v50_lyA = half_velocity(lyA0, dl_lyA_chromo, dl_lyA_abs)
v50_ovi = half_velocity(ovi0, dl_ovi_chromo, dl_ovi_abs)
print(f'Lyα half-dimming speed V50 = {v50_lyA:6.1f} km/s')
print(f'OVI half-dimming speed V50 = {v50_ovi:6.1f} km/s')
print(f'Ratio V50(Lyα)/V50(OVI)    = {v50_lyA / v50_ovi:5.2f} (≈ 4 = sqrt(m_O/m_p))')

## Part 3: Ion Temperature Anisotropy Fit / 이온 온도 비등방성 적합

UVCS observations of polar coronal holes revealed that the perpendicular kinetic temperature $T_\perp$ of O$^{5+}$ ions exceeds the parallel temperature $T_\parallel$ by an order of magnitude — direct evidence for **ion-cyclotron-resonance heating** (paper Sect. 1, Key Takeaway #5). The observable connection is:

- LOS perpendicular to $\vec B$ → measured Doppler width is dominated by $T_\perp$.
- LOS parallel to $\vec B$ (e.g. on-disk near a coronal hole boundary) → width samples $T_\parallel$.

We synthesize two noisy line-profile observations, fit Gaussians, and recover $T_\perp / T_\parallel$. The forward model is:

$$ I(\lambda) = I_0\,\exp\!\left[-\frac{(\lambda-\lambda_0)^2}{\Delta\lambda^2}\right] + \text{background},\qquad \Delta\lambda = \lambda_0 \sqrt{2 k_B T / m_\text{ion}} / c. $$

관측 시선이 자기장에 수직이면 라인 폭은 $T_\perp$, 평행이면 $T_\parallel$를 반영한다. UVCS의 상징적 결과는 빠른 태양풍에서 $T_\perp \gg T_\parallel$로, 본 절에서는 합성 데이터로 두 라인을 가우시안으로 적합하여 비등방성 비를 추정한다.

In [ ]:
def gaussian_with_bg(lam_A, amp, lam0_A, dl_width_A, bg):
    """Gaussian line plus constant background, used as fit model."""
    return amp * np.exp(-((lam_A - lam0_A) / dl_width_A) ** 2) + bg


def width_to_temperature(dl_width_A, lam0_A, m_kg):
    """Invert the thermal Doppler width formula to recover kinetic T."""
    v_th_m_s = (dl_width_A / lam0_A) * (C_LIGHT * 1.0e3)
    return 0.5 * m_kg * v_th_m_s ** 2 / K_B


# Ground truth (Key Takeaway #5 worked example): T_perp = 5e7 K, T_par = 5e6 K.
T_perp_true = 5.0e7
T_par_true = 5.0e6

dl_perp_true = thermal_doppler_width_angstrom(ovi0, T_perp_true, M_OXYGEN)
dl_par_true = thermal_doppler_width_angstrom(ovi0, T_par_true, M_OXYGEN)

lam_axis = np.linspace(ovi0 - 3.0, ovi0 + 3.0, 200)

# Synthetic noisy spectra: LOS perp to B (samples T_perp) and LOS par to B.
y_perp = gaussian_with_bg(lam_axis, 100.0, ovi0, dl_perp_true, 5.0)
y_par = gaussian_with_bg(lam_axis, 100.0, ovi0, dl_par_true, 5.0)
y_perp_noisy = y_perp + rng.normal(0, np.sqrt(np.maximum(y_perp, 0.5)))
y_par_noisy = y_par + rng.normal(0, np.sqrt(np.maximum(y_par, 0.5)))

# Fit Gaussian + bg to each.
p0 = [80.0, ovi0, 0.5, 5.0]
popt_perp, _ = curve_fit(gaussian_with_bg, lam_axis, y_perp_noisy, p0=p0)
popt_par, _ = curve_fit(gaussian_with_bg, lam_axis, y_par_noisy, p0=p0)

T_perp_fit = width_to_temperature(popt_perp[2], ovi0, M_OXYGEN)
T_par_fit = width_to_temperature(popt_par[2], ovi0, M_OXYGEN)

print(f'True  T_perp = {T_perp_true:.2e} K   |  Fit T_perp = {T_perp_fit:.2e} K')
print(f'True  T_par  = {T_par_true:.2e} K   |  Fit T_par  = {T_par_fit:.2e} K')
print(f'True  T_perp/T_par = {T_perp_true / T_par_true:.2f}   |  Fit ratio = {T_perp_fit / T_par_fit:.2f}')

fig, ax = plt.subplots(figsize=(11, 6))
ax.errorbar(lam_axis, y_perp_noisy, yerr=np.sqrt(np.maximum(y_perp_noisy, 1)),
            fmt='o', color='tab:red', ms=3, alpha=0.5, label='LOS ⟂ B (synthetic)')
ax.plot(lam_axis, gaussian_with_bg(lam_axis, *popt_perp), '-', color='tab:red',
        label=f'Fit  T_perp = {T_perp_fit:.1e} K')
ax.errorbar(lam_axis, y_par_noisy, yerr=np.sqrt(np.maximum(y_par_noisy, 1)),
            fmt='s', color='tab:blue', ms=3, alpha=0.5, label='LOS ∥ B (synthetic)')
ax.plot(lam_axis, gaussian_with_bg(lam_axis, *popt_par), '-', color='tab:blue',
        label=f'Fit  T_par  = {T_par_fit:.1e} K')
ax.axvline(ovi0, color='k', linestyle=':', alpha=0.4, label='O VI rest')
ax.set_xlabel('Wavelength [Å]')
ax.set_ylabel('Counts (arbitrary)')
ax.set_title('O VI 1031.93 Å line profile — anisotropy fit')
ax.legend()
plt.tight_layout()
plt.show()

## Part 4: Outflow Speed from O VI 1032/1037 Doublet Ratio / OVI 비를 이용한 흐름 속도 역산

The O VI doublet 1031.93 / 1037.61 Å is **the** UVCS velocity diagnostic in the fast-wind acceleration region. In the optically-thin static limit the ratio $R = I_{1032}/I_{1037}$ equals the oscillator-strength ratio $\approx 2.0$. With outflow:

- Both lines dim individually, but
- The 1037 Å line picks up a secondary contribution from the chromospheric **C II 1037.018 Å** photo-pump that Doppler-shifts onto the O VI 1037.613 Å absorber at $V_W \approx 175$ km/s ($\Delta\lambda = 0.595$ Å, paper Sect. 4.3).

So $R(V_W)$ first rises (the 1037 dims faster while C II is still off-resonance), then drops when C II pumping kicks in, producing a non-monotonic curve (paper Fig. 2). Inverting $R$ for $V_W$ requires choosing the correct branch — UVCS does this by combining with the absolute-radiance constraint from the WLC channel.

OVI 1032/1037 doublet 비 $R$는 정적 광박막 극한에서 2.0이지만 흐름이 생기면 두 라인이 다른 속도로 dimming 되고 1037 Å 라인은 C II 1037.018 Å에 의한 펌핑으로 비단조적 변화를 보인다. 본 절에서는 forward 모델로 $R(V_W)$ 곡선을 만들고 합성 측정값으로부터 $V_W$를 역산한다.

In [ ]:
OVI_1032 = 1031.93
OVI_1037 = 1037.61
C_II_1037 = 1037.018
DELTA_OVI_CII = OVI_1037 - C_II_1037   # ≈ 0.595 Å — paper Sect. 4.3.


def ovi_doublet_ratio(v_wind_km_s, dl_chromo_A, dl_abs_A,
                     alpha_cii=0.30, static_ratio=2.0):
    """Forward model for the I_1032 / I_1037 ratio with C II pumping.

    The 1032 line has only direct O VI resonant scattering. The 1037 line
    has both an O VI self-pumping term (dimmed by V_W) and a C II pumping
    term whose offset depends on the velocity-dependent shift of C II onto
    the 1037.613 Å absorber.

    Args:
        v_wind_km_s: Bulk outflow speed (radial component) in km/s.
        dl_chromo_A: Chromospheric source line 1/e width.
        dl_abs_A: Coronal absorber 1/e width.
        alpha_cii: C II / O VI radiance ratio at the chromospheric source.
        static_ratio: Branching ratio in the static optically-thin limit.

    Returns:
        Synthetic intensity ratio I(1032) / I(1037).
    """
    sigma_sq = dl_chromo_A ** 2 + dl_abs_A ** 2
    # Pure O VI dimming for both lines.
    shift_1032 = OVI_1032 * v_wind_km_s / C_LIGHT
    shift_1037 = OVI_1037 * v_wind_km_s / C_LIGHT
    D_1032 = np.exp(-shift_1032 ** 2 / sigma_sq)
    D_1037_self = np.exp(-shift_1037 ** 2 / sigma_sq)
    # C II pumping: the C II line shifts by V_W * lambda_CII / c onto the
    # 1037.613 Å absorber; resonance occurs when the residual offset is zero.
    cii_shift_at_abs = DELTA_OVI_CII - C_II_1037 * v_wind_km_s / C_LIGHT
    D_1037_cii = alpha_cii * np.exp(-cii_shift_at_abs ** 2 / sigma_sq)
    I_1032 = D_1032
    I_1037 = (D_1037_self + D_1037_cii) / static_ratio
    return I_1032 / I_1037


v_grid = np.linspace(0, 500, 400)
ratio_curve = ovi_doublet_ratio(v_grid, dl_ovi_chromo, dl_ovi_abs,
                                alpha_cii=0.30, static_ratio=2.0)

# Synthetic UVCS measurement: V_true = 220 km/s with 5% Gaussian noise.
V_TRUE = 220.0
ratio_obs = ovi_doublet_ratio(V_TRUE, dl_ovi_chromo, dl_ovi_abs,
                              alpha_cii=0.30, static_ratio=2.0)
ratio_obs_noisy = ratio_obs * (1.0 + 0.05 * rng.standard_normal())

# Invert via root-finding on the high-V branch (V > peak).
v_peak = v_grid[np.argmax(ratio_curve)]
func = lambda v: ovi_doublet_ratio(v, dl_ovi_chromo, dl_ovi_abs,
                                   alpha_cii=0.30, static_ratio=2.0) - ratio_obs_noisy
v_recovered = brentq(func, v_peak + 1, 500)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(v_grid, ratio_curve, lw=2, label='Forward model R(V_W)')
ax.axhline(2.0, color='gray', linestyle=':', label='Static ratio = 2')
ax.axvline(v_peak, color='tab:green', linestyle='--', alpha=0.5,
           label=f'Peak @ V_W ≈ {v_peak:.0f} km/s')
ax.axhline(ratio_obs_noisy, color='tab:red', linestyle='--',
           label=f'Observed R = {ratio_obs_noisy:.2f}')
ax.axvline(V_TRUE, color='black', linestyle=':', label=f'True V_W = {V_TRUE:.0f} km/s')
ax.axvline(v_recovered, color='tab:red', linestyle=':',
           label=f'Recovered V_W = {v_recovered:.0f} km/s')
ax.set_xlabel('Outflow speed V_W [km/s]')
ax.set_ylabel('I(1032) / I(1037)')
ax.set_title('O VI doublet ratio with C II pumping (Kohl+95 Fig. 2 analogue)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'True V_W      = {V_TRUE:6.1f} km/s')
print(f'Observed R    = {ratio_obs_noisy:6.3f}')
print(f'Recovered V_W = {v_recovered:6.1f} km/s   (err = {v_recovered - V_TRUE:+.1f} km/s)')

### 4.1 Branch-degeneracy and joint diagnostics / 분기 모호성과 결합 진단

Because $R(V_W)$ is non-monotonic, a single ratio measurement gives **two** candidate $V_W$ values (one below, one above the peak). UVCS resolves the degeneracy by combining the OVI doublet ratio with the **absolute** Lyα or OVI radiance, normalized to the cospatial WLC electron density (Eq. 2 of the paper). This is exactly the role of the WLC visible polarimetry channel: it removes $N_e$ as a free parameter, so the residual nonlinear inversion has a unique solution.

$R(V_W)$가 비단조이므로 비 측정 한 번만으로는 흐름 속도가 두 값으로 모호하게 결정된다. UVCS는 WLC 가시광 편광계가 측정한 전자 밀도($N_e$)를 입력으로 하여 절대 자외선 라인 강도(식 2)와 함께 결합 진단을 수행함으로써 이 모호성을 해소한다. 이것이 UVCS가 자외선 분광기·가시광 편광계를 한 시선에서 동시에 가져야 하는 이유다.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 (Kohl+1995) | Modern Equivalent / 현대 동등물 |
|---|---|---|
| External + internal occulter geometry / 외부+내부 차폐 기하 | UVCS triple-stage occulter, Eq. (3) unvignetted area | Solar Orbiter Metis externally occulted; PROBA-3/ASPIICS formation-flying occulter |
| H I Lyα profile diagnostic / Lyα 프로파일 진단 | UVCS Lyα channel (1216 Å) — proton thermal width | LASCO-C2 lacks UV; SUMER provides on-disk Lyα; Metis Lyα channel inherits this |
| O VI 1032/1037 doublet ratio / OVI 비 | UVCS OVI channel — V_W via C II pumping (Fig. 2) | Solar Orbiter SPICE off-limb OVI; Metis HeII 30.4 nm complement |
| Ion temperature anisotropy T_⊥ vs T_∥ / 이온 온도 비등방성 | UVCS coronal-hole observations show T_⊥(O⁵⁺) ≫ T_∥ | PSP FIELDS + SWEAP confirm at 0.05 AU; ion-cyclotron heating paradigm |
| Outflow speed retrieval via Doppler dimming / Doppler-dimming 흐름 속도 | OVI ratio + WLC N_e joint inversion | Hinode EIS on-disk; Metis Doppler-dimming maps; Aditya-L1 SoLEXS |

### Key Numerical Results / 핵심 수치 결과

- Unvignetted area at $r = 2.5\,R_\odot$: $A \approx 5{,}500$ mm$^2$ (UV channels). / 2.5 $R_\odot$에서 비비네팅 면적은 약 5500 mm².
- Half-dimming speed: $V_{50}(\text{Lyα}) / V_{50}(\text{OVI}) \approx 4 \approx \sqrt{m_O/m_p}$. / 절반-dimming 속도 비는 약 4 = √(m_O/m_p).
- Anisotropy fit recovers $T_\perp / T_\parallel \approx 10$ from synthetic noisy line profiles, matching the worked example in notes Sect. 4.8.
- O VI ratio inversion recovers $V_W = 220 \pm 10$ km/s on the high-velocity branch — consistent with UVCS coronal-hole observations of the fast wind at 2 R⊙.

### Caveats / 주의 사항

1. The forward model assumes Gaussian chromospheric and absorber profiles. Real UVCS analyses use measured solar disk profiles (Curdt et al. 2001) and bi-Maxwellian absorbers. / 실제 UVCS 해석은 SUMER 등의 색구권 측정 라인 프로파일과 비-맥스웰 흡수 프로파일을 사용한다.
2. We ignore line-of-sight integration: only a single point along the LOS is modelled. The true emergent profile is an LOS-weighted integral of $N_e\,N_\text{ion}$ over the inhomogeneous corona. / 실제는 시선 방향 적분이 필요하다.
3. The C II/O VI pumping coefficient $\alpha_\text{CII} = 0.30$ is illustrative; UVCS data analysis uses radiometrically calibrated chromospheric C II radiance from SUMER. / α_CII는 예시 값이며 실제 분석은 라디오메트리 보정된 값을 사용한다.

## References / 참고문헌

- Kohl, J. L., et al., *Solar Physics* **162**, 313–356 (1995). DOI: 10.1007/BF00733433
- Noci, G., Kohl, J. L., Withbroe, G. L., *Astrophys. J.* **315**, 706 (1987).
- Withbroe, G. L., et al., *Astrophys. J.* **254**, 361 (1982).
- Cranmer, S. R., *Living Reviews in Solar Physics* **6**, 3 (2009).